# Feture engeniring 

In [31]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
from wordcloud import WordCloud , STOPWORDS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load both datasets from the data clansing

In [7]:
tv_ = pd.read_pickle("tv_show.pkl")
tv = pd.read_pickle("tv_show_cleaned.pkl")
tv_fit = pd.read_pickle("tv_for_features.pkl")

# Start with handling the overview column

In [13]:
#tv_['overview'].info()
#tv_.info()
tv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168639 entries, 0 to 168638
Data columns (total 24 columns):
 #   Column                Non-Null Count   Dtype   
---  ------                --------------   -----   
 0   id                    168639 non-null  int64   
 1   number_of_seasons     168639 non-null  int64   
 2   number_of_episodes    168639 non-null  int64   
 3   original_language     168639 non-null  category
 4   vote_count            168639 non-null  int64   
 5   vote_average          168639 non-null  float64 
 6   overview              93333 non-null   string  
 7   adult                 168639 non-null  bool    
 8   in_production         168639 non-null  bool    
 9   popularity            168639 non-null  float64 
 10  type                  168639 non-null  category
 11  status                168639 non-null  category
 12  languages             168639 non-null  int64   
 13  networks              168639 non-null  int64   
 14  origin_country        168639 non-nul

In [22]:
from textblob import TextBlob, Word, Blobber

def polarity(text):
    if not pd.isna(text):
        return TextBlob(text).sentiment.polarity
    else:
        return text
    
def subjectivity(text):
    if not pd.isna(text):
        return TextBlob(text).sentiment.subjectivity
    else:
        return text
    
    
tv_['polarity'] = tv_['overview'].apply(polarity)
tv_['subjectivity'] = tv_['overview'].apply(subjectivity)

tv['polarity'] = tv['overview'].apply(polarity)
tv['subjectivity'] = tv['overview'].apply(subjectivity)

In [19]:
tv_['polarity'].head(20)

0    -0.080000
1    -0.046032
2    -0.231548
3     0.091667
4    -0.351111
5     0.144444
6     0.055000
7     0.047273
8    -0.011111
9     0.000000
10    0.078193
11    0.016667
12    0.075000
13    0.016667
14    0.000000
15   -0.060000
16    0.231111
17    0.500000
18    0.100000
19    0.200000
Name: polarity, dtype: float64

In [25]:
tv_['sentiment'] = tv_['polarity'].apply(lambda x: np.nan if pd.isna(x) else 1 if x > 0 else -1 if x < 0 else 0)
tv['sentiment'] = tv['polarity'].apply(lambda x: np.nan if pd.isna(x) else 1 if x > 0 else -1 if x < 0 else 0)
tv['sentiment'].head()

0   -1.0
1   -1.0
2   -1.0
3    1.0
4   -1.0
Name: sentiment, dtype: float64

In [26]:
tv_.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168639 entries, 0 to 168638
Data columns (total 27 columns):
 #   Column                Non-Null Count   Dtype   
---  ------                --------------   -----   
 0   id                    168639 non-null  int64   
 1   number_of_seasons     168639 non-null  int64   
 2   number_of_episodes    168639 non-null  int64   
 3   original_language     168639 non-null  category
 4   vote_count            168639 non-null  int64   
 5   vote_average          168639 non-null  float64 
 6   overview              93333 non-null   string  
 7   adult                 168639 non-null  bool    
 8   in_production         168639 non-null  bool    
 9   popularity            168639 non-null  float64 
 10  type                  168639 non-null  category
 11  status                168639 non-null  category
 12  languages             110050 non-null  category
 13  networks              97589 non-null   category
 14  origin_country        168639 non-nul

## Extract the top 10 most used words for the top 10 most popular shows after deducting the most frequent amongs the bottom 50% popularity values

In [61]:
from collections import Counter
top_popularity = tv_['overview'][tv_['popularity'] < tv_['popularity'].quantile(0.5)]
exlude = ['series', 'show', 'will','new']# 'two','television','aired','one','school','young','american','life','world','life']
# 1. Combine all overviews into a single string
text = " ".join(top_popularity.dropna().astype(str))

# 2. Optionally clean the text (lowercase, remove punctuation etc.)
text = text.lower()

# 3. Define stopwords to exclude common words
stopwords = set(STOPWORDS).union(set(exlude))

# 4. Split text into words and filter out stopwords
words = [word for word in text.split() if word not in stopwords  and word.isalpha()]

# 5. Count word frequencies
word_freq = Counter(words)

# 6. Get the top 10 most common words
tail_10_words = word_freq.most_common(15)

# 7. Convert to DataFrame for display
tail_words_df = pd.DataFrame(tail_10_words, columns=["word", "frequency"])

tail_15_words = tail_words_df['word'].values
tail_15_words = list(tail_15_words)

print(tail_15_words)

#print(top_words_df)

#top_popularity.head()

['television', 'aired', 'one', 'first', 'produced', 'broadcast', 'tv', 'program', 'hosted', 'two', 'world', 'life', 'story', 'american', 'news']


In [62]:
top_popularity = tv_['overview'][tv_['popularity'] > tv_['popularity'].quantile(0.85)]
exlude = ['series', 'show', 'will','new', 'two','television','aired','one','school','young']
exlude.extend(tail_15_words)
# 1. Combine all overviews into a single string
text = " ".join(top_popularity.dropna().astype(str))

# 2. Optionally clean the text (lowercase, remove punctuation etc.)
text = text.lower()

# 3. Define stopwords to exclude common words
stopwords = set(STOPWORDS).union(set(exlude))

# 4. Split text into words and filter out stopwords
words = [word for word in text.split() if word not in stopwords  and word.isalpha()]

# 5. Count word frequencies
word_freq = Counter(words)

# 6. Get the top 10 most common words
top_10_words = word_freq.most_common(15)

# 7. Convert to DataFrame for display
top_words_df = pd.DataFrame(top_10_words, columns=["word", "frequency"])
print(top_words_df)
top_15_words = top_words_df['word'].values

        word  frequency
0     family       1798
1       love       1655
2      drama       1379
3      three       1270
4      based       1270
5      years       1251
6      lives       1231
7       time       1166
8       high       1131
9       find       1123
10    around       1080
11    people       1049
12  episodes       1031
13       set       1027
14      best        999


# Creat a new top words column

In [63]:
tv_['top_words'] = 0
tv['top_words'] = 0

######
tv_['top_words'] = tv_['overview'].apply(lambda x: np.nan if pd.isna(x) else 1 if any(word in x.lower().split() for word in top_15_words) else 0)
tv['top_words'] = tv['overview'].apply(lambda x: np.nan if pd.isna(x) else 1 if any(word in x.lower().split() for word in top_15_words) else 0)

#tv_['top_words'].head(30)
tv_.info()
tv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168639 entries, 0 to 168638
Data columns (total 28 columns):
 #   Column                Non-Null Count   Dtype   
---  ------                --------------   -----   
 0   id                    168639 non-null  int64   
 1   number_of_seasons     168639 non-null  int64   
 2   number_of_episodes    168639 non-null  int64   
 3   original_language     168639 non-null  category
 4   vote_count            168639 non-null  int64   
 5   vote_average          168639 non-null  float64 
 6   overview              93333 non-null   string  
 7   adult                 168639 non-null  bool    
 8   in_production         168639 non-null  bool    
 9   popularity            168639 non-null  float64 
 10  type                  168639 non-null  category
 11  status                168639 non-null  category
 12  languages             110050 non-null  category
 13  networks              97589 non-null   category
 14  origin_country        168639 non-nul

## Lengh of overview column

In [ ]:
def word_len(row):
    if pd.isna(row):  # Handle NaN values
        return 0 # instead of nan returns 0
    return len(row.split(' '))  # Count words
    
tv_['overview_length'] = tv_['overview'].apply(word_len)
tv['overview_length'] = tv['overview'].apply(word_len)

tv_.info()
tv.info()
#print(tv_['overview_length'].isnull().sum()) # = 0
#print(tv_['popularity'].isnull().sum()) # = 0



#tv_cop = tv_.loc[tv_['overview_length'] !=-0]
# Scatter plot (corrected)
#sns.scatterplot(data=tv_cop, x='overview_length', y='popularity')
#plt.show()
#sns.scatterplot(data=tv_, x='overview_length', y='popularity')
#plt.show()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168639 entries, 0 to 168638
Data columns (total 29 columns):
 #   Column                Non-Null Count   Dtype   
---  ------                --------------   -----   
 0   id                    168639 non-null  int64   
 1   number_of_seasons     168639 non-null  int64   
 2   number_of_episodes    168639 non-null  int64   
 3   original_language     168639 non-null  category
 4   vote_count            168639 non-null  int64   
 5   vote_average          168639 non-null  float64 
 6   overview              93333 non-null   string  
 7   adult                 168639 non-null  bool    
 8   in_production         168639 non-null  bool    
 9   popularity            168639 non-null  float64 
 10  type                  168639 non-null  category
 11  status                168639 non-null  category
 12  languages             168639 non-null  int64   
 13  networks              168639 non-null  int64   
 14  origin_country        168639 non-nul

# Adding columns for the number of spoken langauge, production_countries ...
this is the original data on which I added those new columns before doing the column grouping and cleaning.
I counted the number of instances per specific column such as spoken language, production countries for example: the number of production countries in producing the show 
if for example there are multiple production countries in a show this could imply that the show has high budget and a high budget show coul imply for a popular show.  
  


In [70]:
#this is the original data on which i added those new columns before doing the column grouping and cleaning

#tv_fit.info()
columns = [col for col in tv_fit.columns if col.endswith('_num')]
print(columns) 
for col in columns:
    tv_[col] = tv_fit[col]
    tv[col] = tv_fit[col]   

#tv_.info()
tv.info()

['languages_num', 'spoken_languages_num', 'production_countries_num', 'networks_num']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168639 entries, 0 to 168638
Data columns (total 33 columns):
 #   Column                    Non-Null Count   Dtype   
---  ------                    --------------   -----   
 0   id                        168639 non-null  int64   
 1   number_of_seasons         168639 non-null  int64   
 2   number_of_episodes        168639 non-null  int64   
 3   original_language         168639 non-null  category
 4   vote_count                168639 non-null  int64   
 5   vote_average              168639 non-null  float64 
 6   overview                  93333 non-null   string  
 7   adult                     168639 non-null  bool    
 8   in_production             168639 non-null  bool    
 9   popularity                168639 non-null  float64 
 10  type                      168639 non-null  category
 11  status                    168639 non-null  category
 12  

In [78]:
tv_.head(10)
print(tv_['year_start'].iloc[6])
print(tv_['year_end'].iloc[6])
print(tv_['month_start'].iloc[6])
print(tv_['month_end'].iloc[6])

2021.0
2021.0
9.0
9.0


# Air time column
The duration in which the show been played 

In [72]:
tv_.head(20)
#Adding column for period of each show and removing the date time 


def aired_t(row):
    if pd.isna(row['year_start']) or pd.isna(row['year_end']) or \
           pd.isna(row['month_start']) or pd.isna(row['month_end']):
            return pd.NA  # Keep NaN values
    
    aired = (row['year_end'] - row['year_start']) * 12 + (row['month_end'] - row['month_start'])
    return aired / 12.0

tv_['aired_time'] =tv_.apply(aired_t, axis=1)
tv['aired_time'] =tv.apply(aired_t, axis=1)
#tv_['aired_time'] = (tv_['last_air_year'] - tv_['first_air_year']) *12 + (tv_['last_air_month'] - tv_['first_air_month'])
#tv_['aired_time']=tv_['aired_time']/12

tv_['aired_time'].head(30)

0      8.083333
1      4.583333
2           6.0
3     12.083333
4      5.666667
5      6.583333
6           0.0
7      5.666667
8      5.666667
9      0.166667
10     8.583333
11    11.666667
12     0.083333
13    18.166667
14     3.416667
15    33.833333
16     2.666667
17     3.333333
18     8.583333
19     4.083333
20          3.0
21          9.0
22          7.5
23     0.083333
24    10.083333
25          6.5
26          0.0
27     9.666667
28    15.166667
29     4.666667
Name: aired_time, dtype: object

## Average episodes per season

In [144]:
# Average episode per season 
tv_['avg_episodes_per_season'] = tv_['number_of_episodes'] / tv_['number_of_seasons']
tv['avg_episodes_per_season'] = tv['number_of_episodes'] / tv['number_of_seasons']
tv_['avg_episodes_per_season'].head(30)

0      9.125000
1     13.666667
2      8.500000
3     16.090909
4     15.500000
5     19.571429
6      4.500000
7     12.400000
8     19.333333
9      9.000000
10    20.444444
11    23.250000
12     6.000000
13    22.052632
14     8.000000
15    21.771429
16     8.000000
17     8.000000
18     6.000000
19     7.000000
20     8.000000
21    10.142857
22    21.375000
23     6.000000
24    25.000000
25    14.285714
26     4.000000
27    22.800000
28    21.800000
29     8.000000
Name: avg_episodes_per_season, dtype: float64

# Drop the overview column I extracted what I neded from it 

In [146]:

#tv.drop('overview', axis=1, inplace=True)
#tv_.drop('overview', axis=1, inplace=True)
#tv_.info()
tv.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168639 entries, 0 to 168638
Data columns (total 34 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   id                        168639 non-null  int64  
 1   number_of_seasons         168639 non-null  int64  
 2   number_of_episodes        168639 non-null  int64  
 3   original_language         168639 non-null  int8   
 4   vote_count                168639 non-null  int64  
 5   vote_average              168639 non-null  float64
 6   adult                     168639 non-null  int64  
 7   in_production             168639 non-null  int64  
 8   popularity                168639 non-null  float64
 9   type                      168639 non-null  int8   
 10  status                    168639 non-null  int8   
 11  languages                 168639 non-null  int64  
 12  networks                  168639 non-null  int64  
 13  origin_country            168639 non-null  i

In [148]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
for col in tv_.select_dtypes(include= ['object','string','category']):
    tv_[col] = encoder.fit_transform(tv_[col])
    
for col in tv.select_dtypes(include= ['object','string','category']):  
    tv[col] = encoder.fit_transform(tv[col])

for col in tv.select_dtypes(include= [ float, int]):
    tv[col] = tv[col].fillna(0)

for col in tv_.select_dtypes(include= [ float, int]):
    tv_[col] = tv_[col].fillna(0)

    

# Check the coding of the columns and devid the data 

In [94]:
#tv_.info()
for col in tv.select_dtypes(include='bool'):
    tv_[col] = tv_[col].astype(int)
    tv[col] = tv[col].astype(int)

for col in tv_.select_dtypes(include='object'):
    tv_[col] = tv_[col].astype('category')
    tv[col] = tv[col].astype('category')
    

for col in tv.select_dtypes(include='category'):
    tv_[col] = tv_[col].cat.codes
    tv[col] = tv[col].cat.codes

tv.info()
#tv_.info()  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168639 entries, 0 to 168638
Data columns (total 33 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   id                        168639 non-null  int64  
 1   number_of_seasons         168639 non-null  int64  
 2   number_of_episodes        168639 non-null  int64  
 3   original_language         168639 non-null  int8   
 4   vote_count                168639 non-null  int64  
 5   vote_average              168639 non-null  float64
 6   adult                     168639 non-null  int64  
 7   in_production             168639 non-null  int64  
 8   popularity                168639 non-null  float64
 9   type                      168639 non-null  int8   
 10  status                    168639 non-null  int8   
 11  languages                 168639 non-null  int64  
 12  networks                  168639 non-null  int64  
 13  origin_country            168639 non-null  i

In [149]:
x_ = tv_.drop('popularity', axis=1)
y_ = tv_['popularity']

x = tv.drop('popularity', axis=1)
y = tv['popularity']

x.info()
x_.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168639 entries, 0 to 168638
Data columns (total 33 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   id                        168639 non-null  int64  
 1   number_of_seasons         168639 non-null  int64  
 2   number_of_episodes        168639 non-null  int64  
 3   original_language         168639 non-null  int8   
 4   vote_count                168639 non-null  int64  
 5   vote_average              168639 non-null  float64
 6   adult                     168639 non-null  int64  
 7   in_production             168639 non-null  int64  
 8   type                      168639 non-null  int8   
 9   status                    168639 non-null  int8   
 10  languages                 168639 non-null  int64  
 11  networks                  168639 non-null  int64  
 12  origin_country            168639 non-null  int8   
 13  spoken_languages          168639 non-null  i

In [135]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

In [116]:
pd.set_option('display.max_columns', None)
x.head()
print(x.isnull().sum())

id                          0
number_of_seasons           0
number_of_episodes          0
original_language           0
vote_count                  0
vote_average                0
adult                       0
in_production               0
type                        0
status                      0
languages                   0
networks                    0
origin_country              0
spoken_languages            0
production_countries        0
episode_run_time            0
changed_name                0
year_start                  0
month_start                 0
year_end                    0
month_end                   0
group_genere                0
polarity                    0
subjectivity                0
sentiment                   0
top_words                   0
overview_length             0
languages_num               0
spoken_languages_num        0
production_countries_num    0
networks_num                0
aired_time                  0
dtype: int64


# I will use Gridsearch to find the best alpha for lasso and ridge model

In [136]:
# define alpha search space
alpha = {'alpha': np.logspace(-3, 2, 20)}
#LASSO
lasso_search = GridSearchCV(Lasso(max_iter=10000), alpha, cv=5)
lasso_search.fit(x, y)
best_lasso = lasso_search.best_estimator_
print("Best alpha for Lasso:", lasso_search.best_params_)

# Grid Search for Ridge
ridge_search = GridSearchCV(Ridge(), alpha, cv=5)
ridge_search.fit(x, y)

best_ridge = ridge_search.best_estimator_
print("Best alpha for Ridge:", ridge_search.best_params_)


Best alpha for Lasso: {'alpha': np.float64(54.555947811685144)}
Best alpha for Ridge: {'alpha': np.float64(100.0)}


The best found parameters are  lasso :54.44 , ridge : 100.
this alpha (especially for lasso ) yields very few columns left therfore i will reduce the lasso alpha 

In [150]:
lasso = Lasso(alpha=5).fit(x, y)
lasso_selected = (np.abs(lasso.coef_) > 0).astype(int)

#Ridge
ridge = Ridge(alpha=100).fit(x, y)
ridge_selected = (np.abs(ridge.coef_) > 0).astype(int)

#Gradient Boosting
gb = GradientBoostingRegressor().fit(x, y)
gb_selected = (gb.feature_importances_ > 0).astype(int)

#Random Forest
rf = RandomForestRegressor().fit(x, y)
rf_selected = (rf.feature_importances_ > 0).astype(int)

# Create a DataFrame with the selected features
selected_features = pd.DataFrame({
    'Feature': x.columns,
    'Lasso': lasso_selected,
    'Ridge': ridge_selected,
    'GB': gb_selected,
    'RF': rf_selected
})

# Sum the totals for each model
selected_features['Total'] = selected_features[['Lasso','Ridge','GB','RF']].sum(axis=1)
print(selected_features)

ValueError: Input X contains infinity or a value too large for dtype('float64').

# for the data without the nan fills

In [132]:
lasso = Lasso(alpha=5).fit(x_, y_)
lasso_selected = (np.abs(lasso.coef_) > 0).astype(int)

#Ridge
ridge = Ridge(alpha=5).fit(x_, y_)
ridge_selected = (np.abs(ridge.coef_) > 0).astype(int)

#Gradient Boosting
gb = GradientBoostingRegressor().fit(x_, y_)
gb_selected = (gb.feature_importances_ > 0).astype(int)

#Random Forest
rf = RandomForestRegressor().fit(x_, y_)
rf_selected = (rf.feature_importances_ > 0).astype(int)

# Create a DataFrame with the selected features
selected_features_ = pd.DataFrame({
    'Feature': x_.columns,
    'Lasso': lasso_selected,
    'Ridge': ridge_selected,
    'GB': gb_selected,
    'RF': rf_selected
})

# Sum the totals for each model
selected_features_['Total'] = selected_features[['Lasso','Ridge','GB','RF']].sum(axis=1)
print(selected_features)

                     Feature  Lasso  Ridge  GB  RF  Total
0                         id      1      1   1   1      4
1          number_of_seasons      0      1   1   1      3
2         number_of_episodes      1      1   1   1      4
3          original_language      0      1   1   1      3
4                 vote_count      1      1   1   1      4
5               vote_average      1      1   1   1      4
6                      adult      0      1   0   1      2
7              in_production      0      1   0   1      2
8                       type      0      1   1   1      3
9                     status      0      1   1   1      3
10                 languages      1      1   1   1      4
11                  networks      1      1   1   1      4
12            origin_country      1      1   1   1      4
13          spoken_languages      1      1   1   1      4
14      production_countries      1      1   1   1      4
15          episode_run_time      0      1   1   1      3
16            

In [147]:
pd.set_option('display.max_columns', None)
print(selected_features)
#print(selected_features[selected_features['Total']==4].sum())
print(selected_features[selected_features['Total']==4].value_counts().sum())
# tv_
#print(selected_features_[selected_features_['Total']==4].value_counts().sum())
#print(len(selected_features[selected_features['Total']==4]))

                     Feature  Lasso  Ridge  GB  RF  Total
0                         id      1      1   1   1      4
1          number_of_seasons      0      1   1   1      3
2         number_of_episodes      1      1   1   1      4
3          original_language      0      1   1   1      3
4                 vote_count      1      1   1   1      4
5               vote_average      0      1   1   1      3
6                      adult      0      1   0   1      2
7              in_production      0      1   0   1      2
8                       type      0      1   1   1      3
9                     status      0      1   1   1      3
10                 languages      1      1   1   1      4
11                  networks      1      1   1   1      4
12            origin_country      1      1   1   1      4
13          spoken_languages      1      1   1   1      4
14      production_countries      0      1   1   1      3
15          episode_run_time      1      1   1   1      4
16            

# Creating DataFrame with most valuable variables
I am creating 2 df 1 without the NAN imputation to check if the imputation in this case harm the final resulted 



In [ ]:
final_col_lst = selected_features[selected_features['Total']==4]['Feature'].to_list()
final_col_lst_ = selected_features_[selected_features_['Total']==4]['Feature'].to_list()
df_model = tv[final_col_lst].copy()
df_model_ = tv_[final_col_lst_].copy()

df_model.info()